# Computação Quântica na Prática — Nível Intermediário

Neste nível o foco passa da manipulação de qubits para a **lógica dos algoritmos
quânticos**. Os temas abordados são:

1. Sistemas de **múltiplos qubits** e o produto tensorial;
2. **Emaranhamento** de muitos corpos (estado GHZ);
3. **Interferência** — o mecanismo que confere vantagem aos algoritmos;
4. **Phase kickback**, um padrão recorrente em algoritmos;
5. **Deutsch–Jozsa** — primeiro algoritmo com vantagem demonstrável;
6. **Bernstein–Vazirani** — recuperação de uma informação oculta em uma consulta;
7. **Busca de Grover** — aceleração quadrática, analisada em detalhe.

Os exercícios avaliados prosseguem: informe a resposta e chame `verificar(...)`.

**Duração estimada:** de 2 a 3 horas.


In [ ]:
# %pip install qiskit qiskit-aer qiskit-ibm-runtime matplotlib pylatexenc seaborn
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_state_qsphere
import qutils
from grader import verificar, progresso
print("Ambiente carregado com sucesso.")

---
## 1. Sistemas de múltiplos qubits e o produto tensorial

O estado de $N$ qubits é descrito por um vetor com $2^N$ amplitudes, uma para cada
combinação da base computacional. Para dois qubits, a base é
$\{|00\rangle, |01\rangle, |10\rangle, |11\rangle\}$.

Quando os qubits são **independentes**, o estado conjunto é o **produto tensorial**
dos estados individuais. Por exemplo, aplicar Hadamard a cada um de dois qubits
produz superposição uniforme sobre os quatro estados, cada um com probabilidade
$1/4$.

> **Convenção de ordenação:** o Qiskit numera os qubits a partir de 0 e, nas
> bitstrings, o qubit 0 é o dígito **mais à direita**.

### Exercício 1 — superposição em dois qubits

Construa um circuito de dois qubits com uma Hadamard em cada qubit (0 e 1).

In [ ]:
sup2 = QuantumCircuit(2)

# Insira abaixo: uma Hadamard no qubit 0 e outra no qubit 1


sup2.measure_all()
verificar("inter-1", sup2)

In [ ]:
estado = Statevector.from_instruction(sup2.remove_final_measurements(inplace=False))
print("Vetor de estado (4 amplitudes):", np.round(estado.data, 3))
print("Probabilidades:", qutils.probabilidades(qutils.simular(sup2, shots=4000)))
qutils.plotar(qutils.simular(sup2, shots=4000), titulo="Superposição uniforme em 2 qubits (~25% cada)")

---
## 2. Emaranhamento de muitos corpos: o estado GHZ

O estado **GHZ** (Greenberger–Horne–Zeilinger) generaliza o estado de Bell para
três qubits, mantendo os três mutuamente emaranhados. A construção estende a receita
do Bell:

1. Hadamard no qubit 0;
2. CNOT do qubit 0 para o 1;
3. CNOT do qubit 1 para o 2.

Diferentemente do estado do Módulo 1 (um produto de estados independentes), o GHZ
**não** pode ser fatorado em qubits separados — é genuinamente emaranhado.

### Exercício 2 — construção do estado GHZ (3 qubits)

In [ ]:
ghz = QuantumCircuit(3)

# Insira abaixo (3 portas):
# 1) Hadamard no qubit 0
# 2) CNOT com controle 0 e alvo 1
# 3) CNOT com controle 1 e alvo 2


ghz.measure_all()
ghz.draw("mpl")

In [ ]:
verificar("inter-2", ghz)

In [ ]:
estado = Statevector.from_instruction(ghz.remove_final_measurements(inplace=False))
print("Amplitudes não nulas apenas em |000> e |111>:", np.round(estado.data, 3))
plot_state_qsphere(estado)

### Exercício 3 — resultados do estado GHZ

Ao medir o estado GHZ repetidamente, quais resultados podem ser observados? Responda
como um conjunto, por exemplo `{"000", "111"}`.

In [ ]:
resposta = set()   # informe o conjunto
verificar("inter-3", resposta)

---
## 3. Interferência: o mecanismo dos algoritmos quânticos

A superposição, isoladamente, equivale a um sorteio aleatório — não conferiria
vantagem alguma. O elemento decisivo é a **interferência**: assim como ondas, as
amplitudes quânticas podem **somar-se** (interferência construtiva) ou **cancelar-se**
(destrutiva). Um algoritmo quântico é projetado de modo que as amplitudes das
respostas incorretas se cancelem e as das corretas se reforcem.

O exemplo mais simples é a sequência **H–Z–H**. A primeira Hadamard cria
superposição; a porta Z inverte a fase de $|1\rangle$; a segunda Hadamard faz as
amplitudes interferirem. O resultado é o estado $|1\rangle$ com certeza — ou seja,
$H\,Z\,H = X$. A fase, invisível a uma medição intermediária, determina
integralmente o resultado final.

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)     # superposição
qc.z(0)     # inverte a fase de |1>
qc.h(0)     # interferência
estado = Statevector.from_instruction(qc)
print("Estado final:", np.round(estado.data, 3), "-> corresponde a |1>")
print("Probabilidades:", estado.probabilities_dict())
print()
print("Comparação: sem a porta Z, H seguido de H retorna ao estado |0> (interferência")
print("destrutiva sobre |1>). A fase intermediária define o desfecho.")

### Exercício 4 — o papel da interferência

O que distingue um algoritmo quântico de um simples sorteio aleatório sobre estados
em superposição?

- **(a)** O número de qubits utilizados.
- **(b)** A interferência: as amplitudes das respostas erradas se cancelam e as das
  corretas se reforçam.
- **(c)** A velocidade do processador.
- **(d)** O uso de bits clássicos auxiliares.

In [ ]:
resposta = ""   # "a", "b", "c" ou "d"
verificar("inter-4", resposta)

---
## 4. Phase kickback

Um padrão recorrente em algoritmos quânticos é o **phase kickback**: quando uma
porta controlada atua sobre um alvo preparado no estado $|-\rangle$, a fase
associada ao alvo é "transferida" para o qubit de controle.

Concretamente, o estado $|-\rangle$ é um autoestado da porta X com autovalor $-1$:
$X|-\rangle = -|-\rangle$. Assim, ao aplicar um CNOT com o alvo em $|-\rangle$, o
fator $-1$ recai sobre o controle, alterando sua fase em vez do alvo. É esse
mecanismo que Deutsch–Jozsa e Bernstein–Vazirani exploram.

In [ ]:
qc = QuantumCircuit(2)
qc.h(0)                 # controle em |+>
qc.x(1); qc.h(1)        # alvo em |->
qc.cx(0, 1)             # CNOT: a fase -1 recai sobre o CONTROLE (kickback)

estado = Statevector.from_instruction(qc)
# Isola o estado do qubit de controle (qubit 0):
from qiskit.quantum_info import partial_trace, DensityMatrix
print("Amplitudes do estado completo:", np.round(estado.data, 3))
print()
print("O controle passou de |+> a |-> (a fase foi transferida do alvo para o controle),")
print("enquanto o alvo permaneceu em |->. Nenhum bit mudou — apenas a fase do controle.")

---
## 5. Deutsch–Jozsa: vantagem em uma única consulta

**Problema.** Dada uma função-caixa-preta (um *oráculo*) garantidamente **constante**
(retorna sempre o mesmo valor) ou **balanceada** (retorna 0 para metade das entradas
e 1 para a outra metade), determinar a qual categoria ela pertence.

- **Abordagem clássica:** no pior caso, é necessário avaliar metade das entradas
  mais uma.
- **Deutsch–Jozsa:** uma única execução do circuito é suficiente.

A leitura do resultado é direta: se a medição dos qubits de entrada retorna apenas
zeros, a função é constante; caso contrário, é balanceada. A célula abaixo constrói
o circuito para um oráculo constante (que não atua sobre a entrada) e para um
balanceado (um CNOT de cada qubit de entrada para o auxiliar), usando o phase
kickback da seção anterior.

In [ ]:
def deutsch_jozsa(tipo_oraculo, n=3):
    qc = QuantumCircuit(n + 1, n)     # n qubits de entrada + 1 auxiliar
    qc.x(n); qc.h(n)                  # auxiliar em |-> (habilita o phase kickback)
    qc.h(range(n))                    # entrada em superposição
    qc.barrier()
    if tipo_oraculo == "balanceada":
        for q in range(n):
            qc.cx(q, n)               # oráculo balanceado
    # oráculo constante: nenhuma operação
    qc.barrier()
    qc.h(range(n))                    # interferência
    qc.measure(range(n), range(n))
    return qc

for tipo in ("constante", "balanceada"):
    contagens = qutils.simular(deutsch_jozsa(tipo, n=3), shots=1000)
    print("Oráculo {:<11} -> medições: {}".format(tipo, qutils.probabilidades(contagens)))

O oráculo constante produz sempre `000`, enquanto o balanceado produz `111` — em
ambos os casos, com uma única execução.

### Exercício 5 — interpretação do resultado

Se a execução de Deutsch–Jozsa retornar `0...0` (apenas zeros) nos qubits de
entrada, a função é *constante* ou *balanceada*? Responda com uma palavra.

In [ ]:
resposta = ""   # "constante" ou "balanceada"
verificar("inter-5", resposta)

---
## 6. Bernstein–Vazirani: recuperação de uma string oculta

O algoritmo de **Bernstein–Vazirani** resolve um problema correlato. O oráculo
codifica uma **string binária secreta** $s$; o objetivo é descobri-la.

- **Abordagem clássica:** cada consulta revela um bit, exigindo $n$ consultas para
  uma string de $n$ bits.
- **Bernstein–Vazirani:** uma única consulta basta — a string inteira é obtida
  diretamente na medição.

O oráculo aplica um CNOT ao auxiliar (em $|-\rangle$) a partir de cada qubit de
entrada correspondente a um bit 1 de $s$. Pelo phase kickback, a informação de $s$
é transferida para as fases dos qubits de entrada e, após uma camada de Hadamards,
lida diretamente. A célula abaixo usa a string secreta `1011`.

In [ ]:
def bernstein_vazirani(s):
    n = len(s)
    qc = QuantumCircuit(n + 1, n)
    qc.x(n); qc.h(n)                       # auxiliar em |->
    qc.h(range(n))                         # entrada em superposição
    qc.barrier()
    for i, bit in enumerate(reversed(s)):  # qubit 0 = bit mais à direita de s
        if bit == "1":
            qc.cx(i, n)                    # oráculo codifica a string secreta
    qc.barrier()
    qc.h(range(n))
    qc.measure(range(n), range(n))
    return qc

contagens = qutils.simular(bernstein_vazirani("1011"), shots=500)
print("Medição (string recuperada):", contagens)

### Exercício 6 — a string recuperada

Execute o circuito acima (string secreta `1011`). Qual bitstring é medida — ou seja,
qual string secreta o algoritmo recupera? Responda com a sequência de bits.

In [ ]:
resposta = ""   # informe a bitstring recuperada
verificar("inter-6", resposta)

---
## 7. Busca de Grover: aceleração quadrática

Considere a busca por um item específico em uma coleção **não ordenada** de $N$
itens. Classicamente, no pior caso, examinam-se os $N$ itens. O algoritmo de
**Grover** localiza o item em cerca de $\sqrt{N}$ passos — uma aceleração
**quadrática**.

A cada iteração aplicam-se dois operadores:

- **Oráculo:** inverte a fase do estado procurado (o "marca"), sem alterar os demais.
- **Difusor:** reflete as amplitudes em torno de sua média, convertendo a marcação
  de fase em aumento de probabilidade (**amplificação de amplitude**).

O número ótimo de iterações é aproximadamente $(\pi/4)\sqrt{N}$. A célula a seguir
implementa uma busca de Grover com 2 qubits ($N=4$) procurando o estado `11`, e
acompanha a amplitude do estado marcado a cada etapa.

In [ ]:
def oraculo_11(qc):
    qc.cz(0, 1)                 # marca |11> com uma fase negativa

def difusor(qc):
    qc.h([0, 1]); qc.x([0, 1]); qc.cz(0, 1); qc.x([0, 1]); qc.h([0, 1])

grover = QuantumCircuit(2)
grover.h([0, 1])                # superposição uniforme sobre os 4 estados
oraculo_11(grover)
difusor(grover)
grover.measure_all()

print("Resultado de Grover:", qutils.probabilidades(qutils.simular(grover, shots=2000)))
qutils.plotar(qutils.simular(grover, shots=2000), titulo="Grover: amplificação de |11>")

Com uma única iteração, o estado `11` já é obtido com probabilidade próxima de 1.

Um ponto importante: **mais iterações não significam melhor resultado**. A
amplificação é uma rotação no espaço de amplitudes; passar do ponto ótimo faz a
probabilidade do alvo **diminuir** (fenômeno de *over-rotation*). Por isso o número
de iterações deve seguir $(\pi/4)\sqrt{N}$.

### Exercício 7 — número de iterações

Para uma coleção de $N = 16$ itens com uma única solução, qual é o número ótimo de
iterações de Grover? Utilize $(\pi/4)\sqrt{N}$ e arredonde para o inteiro mais
próximo.

In [ ]:
import math
# Sugestão: math.pi e math.sqrt(16)
resposta = 0   # informe o inteiro
verificar("inter-7", resposta)

### Exercício 8 — marcando outro estado

O oráculo determina qual estado é amplificado. Modifique a busca para marcar o
estado `10` em vez de `11` e execute o circuito.

Sugestão: para marcar `10`, aplique portas X que levem `10` a `11` antes do `cz` e
as desfaça em seguida (ou construa o oráculo equivalente). Verifique qual resultado
passa a dominar as medições e responda como conjunto, por exemplo `{"10"}`.

In [ ]:
grover10 = QuantumCircuit(2)
grover10.h([0, 1])

# Construa aqui o oráculo que marca |10> e, em seguida, aplique o difusor.
# (dica: qc.x no qubit adequado, qc.cz(0,1), qc.x de volta; depois o difusor)


grover10.measure_all()
print("Distribuição:", qutils.probabilidades(qutils.simular(grover10, shots=2000)))

# Informe o resultado dominante que você observou:
resposta = set()   # ex.: {"10"}
verificar("inter-8", resposta)

---
## Síntese

Este nível apresentou o que torna os algoritmos quânticos eficazes: **superposição**
para explorar muitas possibilidades e **interferência** (frequentemente via *phase
kickback*) para concentrar a probabilidade nas respostas corretas. Esses princípios
foram aplicados em três algoritmos — Deutsch–Jozsa, Bernstein–Vazirani e Grover.

O notebook **completo** dá continuidade com ruído e mitigação de erros,
teletransporte quântico, algoritmos variacionais (VQE) e aplicações.

Execute a célula abaixo para visualizar o progresso.

In [ ]:
progresso("inter")